<a href="https://colab.research.google.com/github/fl0risk/MNTF_Project/blob/main/MNTF_Floris_Koster.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MNTF Project by Floris Koster 21-926-605

In [ ]:
## NOTE: GPU Supported

## Code

In [ ]:
#import packages
import numpy as np
#import scipy.stats as scipy
#from scipy.stats import norm
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

First, we define the Euler-Scheme to generate paths of processes given by SDE's. In particular, we define a 2-dimensional Euler-Scheme. This 2-dimensionality is used in the generating process for the **Heston model**.

In [ ]:
def euler_scheme(T, dt, R,second_true,x0,drift_x,vol1_x,W1,
                    drift_y =  lambda x,y,t: 0,vol1_y =  lambda x,y,t: 0,vol2_x = lambda x,y,t: 0,vol2_y =  lambda x,y,t: 0,y0 =  None,W2 = None):
  """
  Performs the Euler scheme for a 2D SDE.
  Args:
    T: The final time.
    dt: The time step.
    R: number of generated paths
    second_true: bool if false only one SDE
    x0: The initial value of x.
    driftx: A vector field representing the drift term in Eq.1.
    drifty: A vector field representing the drift term in Eq.2.
    vol1_x: Integrand of dW_1 in the noise term (volatility) in Eq.1.
    W1: R Brownian motion paths for first SDE
    vol2_x: Integrand of dW_1 in the noise term (volatility) in Eq.2.
    vol1_y: Integrand of dW_2 in the noise term (volatility) in Eq.1.
    vol2_y: Integrand of dW_2 in the noise term (volatility) in Eq.2.
    y0: The initial value of y.
    W2: R Brownian motion paths for second SDE
  Returns:
    A tuple containing the time points, x values, and y values.
  """
  N = int(T / dt)
  t = np.linspace(0, T, N+ 1)
  x = np.zeros((R,N+ 1))
  y = np.zeros((R,N+ 1))
  x[:,0] = x0
  if second_true:
    y[:,0] = y0
  for i in range(N):
    if second_true:
      x[:,i + 1] = np.maximum(x[:,i] + drift_x(x[:,i], y[:,i], t[i]) * dt + vol1_x(x[:,i], y[:,i], t[i]) * (W1[:,i+1]-W1[:,i]) + vol1_y(x[:,i], y[:,i], t[i]) * (W2[:,i+1]-W2[:,i]), 0)
      y[:,i + 1] = np.maximum(y[:,i] + drift_y(x[:,i], y[:,i], t[i]) * dt + vol2_x(x[:,i], y[:,i], t[i]) *  (W1[:,i+1]-W1[:,i]) + vol2_y(x[:,i], y[:,i], t[i]) * (W2[:,i+1]-W2[:,i]), 0)
    else:
      x[:,i + 1] = np.maximum(x[:,i] + drift_x(x[:,i], t[i]) * dt + vol1_x(x[:,i], t[i]) * (W1[:,i+1]-W1[:,i]),0)
  return t, x, y


Before continuing with the implementation of the different price model, we first provide the dynamics for each model.
The **Black-Scholes model** is given by the dynamics
$$\mathrm{d}S_t = \mu S_t\mathrm{d}t + \sigma S_t\mathrm{d}W_t, \; S_0 = 1$$

The **Heston models** dynamics is described by the coupled SDEs:
$$
\begin{aligned}
\mathrm{d}S_t &= \mu S_t\mathrm{d}t + \sqrt{\nu_t}S_t\mathrm{d}W_t^{(1)}, \; S_0 = 1 \\
\mathrm{d}\nu_t &= k(\theta - \nu_t)\mathrm{d}t + \xi \sqrt{\nu_t}\mathrm{d}W_t^{(2)}, \; \nu_0 = 0.02
\end{aligned}
$$
where $\mathrm{d}W_t^{(1)}$ and $\mathrm{d}W_t^{(2)}$ are Brownian motions with correlation $\rho$, $\theta$ denotes the long-term variance and $k$ is the rate of reversion.

The **CEV (Constant Elasticity of Variance) model** is given by
$$\mathrm{d}S_t = \mu S_t\mathrm{d}t + \sigma S_t^{\gamma}\mathrm{d}W_t, S_0 = 1$$
where $\gamma>0$ is the elasticity parameter.

In addition to the risky asset, we assume the existence of a risk-free bank account (or bond) with deterministic dynamics
$$\mathrm{d}B_t = rB_t\mathrm{d}t, \; B_0 = 1,$$
where $r$ is the constant risk-free rate.

Our wealth process $X = (X_t)_{t \ge 0}$ evolves according to the chosen trading strategy $\alpha = (\alpha_t)_{t \ge 0}$, where $\alpha_t$ denotes the proportion of wealth invested in the risky asset at time $t$. The remaining wealth is invested in the risk-free asset. The wealth dynamics are given by
$$
\mathrm{d}X_t =  \alpha_t X_t\frac{\mathrm{d}S_t}{S_t} + (1-\alpha_t)X_t r\mathrm{d}t
$$
where $S_t$ is the price of the risky asset, modelled using one of the above models and $X_0 = 1$. We assume that $\alpha_t \in [-1,1]$ since we use $\tanh$ activation.

Using Itô's formula and the above dynamics for the price models we get the following SDE's that are later used in the forward method of our neural networks
\begin{aligned}
\textbf{BS: } \mathrm{d}\log(X_t) &= \alpha_t \frac{\mathrm{d}S_t}{S_t} + (1-\alpha_t)r\mathrm{d}t - \frac{1}{2} \alpha_t^2\sigma^2\mathrm{d}t; \\
\textbf{Heston: } \mathrm{d}\log(X_t) &= \alpha_t \frac{\mathrm{d}S_t}{S_t} + (1-\alpha_t)r\mathrm{d}t - \frac{1}{2} \alpha_t^2\nu_t\mathrm{d}t; \\
\textbf{CEV: } \mathrm{d}\log(X_t) &= \alpha_t \frac{\mathrm{d}S_t}{S_t} + (1-\alpha_t)r\mathrm{d}t - \frac{1}{2} \alpha_t^2\sigma^2 S_t^{\gamma-2}\mathrm{d}t.
\end{aligned}

Our goal is now to train two trading agents (Markovian and PathDependent) that find a process $(\alpha_t)_{t\ge 0}$, which maximizes expected utility for some utility function determined later. For that we will discretize our space by choosing some step-size $\mathrm{d}t >0$ and generate price processes, following either a BS model, Heston model or CEV model, using the Euler method.

In our case we use the following parameter, which are **not** choosen because of some data calibration:

Bank Account
\begin{align}
r = 0.02
\end{align}
Black-Scholes Model:
\begin{align}
\mu &= 0.05\\
\sigma &= 0.2
\end{align}
Heston Model:
\begin{align}
\mu&=0.05, \; \rho=0.7\\
 k&=4, \; \theta=0.02\\
 \xi&=0.9
 \end{align}
 CEV Model:
\begin{align}
\mu &= 0.05\\
\sigma &= 0.2\\
\gamma &= 0.7
\end{align}

First, we construct for each price model (Black-Scholes, Heston, CEV) a class that consists of methods to generate a number of paths ```N_paths```, called ```generate```, and to compute the corresponding incremenents for the paths, called ```incr```. Additionally, the Heston class includes a method to generate two correlated Brownian motions from two independent ones given some correlation $\rho$.

In [ ]:
class BSModel(object):
    def __init__(self,Npaths,T,dt,mu=0.04,sigma=0.2,S0=1):
        self.T = T
        self.mu = mu
        self.sigma = sigma
        self.N = int(T/dt)
        self.dt = dt
        self.Npaths = Npaths
        self.W = None #Paths of BM with discretization int(T/dt) and Npaths different paths
        self.S0_scalar = S0
        self.S0 = S0*np.ones((Npaths,)) #initial price vector
        self.BS = None

    def __generateBM(self, training_seed = None, eval_seed =  None):
        if training_seed is not None:
            np.random.seed(seed=training_seed)
        if eval_seed is not None:
            np.random.seed(seed = eval_seed)
        dW =  np.sqrt(self.dt)*np.random.randn(self.N*self.Npaths).reshape((self.Npaths,self.N))
        W = np.concatenate((np.zeros((self.Npaths,1)),np.cumsum(dW,axis = 1)),axis =1)
        return W
    def generate(self, training_seed = None, eval_seed = None):
        self.W = self.__generateBM(training_seed, eval_seed)
        drift_BS = lambda x,t : self.mu*x
        volatility_BS = lambda x,t: self.sigma*x
        _,BS,_ = euler_scheme(T = self.T,dt = self.dt,R= self.Npaths,second_true = False,x0 = self.S0,drift_x = drift_BS,vol1_x=volatility_BS,W1=self.W)
        self.BS = BS

    def incr(self):
        if self.BS is not None:
            return np.diff(self.BS)
        else:
            raise ValueError("The paths have not been generated yet. Please run the 'generate' method first.")

    def time(self):
        return np.linspace(0, self.T, self.N+ 1)
    def plot(self):
        for j in range(self.Npaths):
            plt.plot(self.time(), self.BS[j, :])
            plt.title("Black-Scholes Model Trajectories")
            plt.xlabel("Time")
            plt.ylabel("Price")
            plt.show()

In [ ]:
class HestonModel(object):
    def __init__(self,Npaths,T,dt,mu=0.04,rho=0.7, k=4, theta=0.02, xi=0.9,S0=1):
        self.T = T
        self.mu = mu
        self.rho = rho
        self.k = k
        self.theta = theta
        self.xi = xi
        self.N = int(T/dt)
        self.dt = dt
        self.Npaths = Npaths
        self.W = None #Paths of BM with discretization int(T/dt) and Npaths different paths
        self.B = None #second brownian motion independent of W
        self.S0_scalar = S0
        self.S0 = S0*np.ones((Npaths,)) #initial price vector
        self.v0 = theta*np.ones((Npaths,)) #initial variance
        self.H = None

    def __generateBM(self, training_seed = None, eval_seed =  None):
        if training_seed is not None:
            np.random.seed(training_seed)
        if eval_seed is not None:
            np.random.seed(eval_seed)
        dW =  np.sqrt(self.dt)*np.random.randn(self.N*self.Npaths).reshape((self.Npaths,self.N))
        W = np.concatenate((np.zeros((self.Npaths,1)),np.cumsum(dW,axis = 1)),axis =1)
        return W
    def __corrBM(self):
        return  self.rho*self.W + np.sqrt(1-self.rho**2)*self.B
    def generate(self, training_seed = None, eval_seed = None):
        self.W = self.__generateBM(training_seed, eval_seed)
        if training_seed is not None:
            training_seed += 1
        if eval_seed is not None:
            eval_seed +=1
        self.B = self.__generateBM(training_seed, eval_seed)
        drift_H_x = lambda x,y,t: self.mu*x
        volH_1_x = lambda x,y,t: np.sqrt(y)*x
        drift_H_y = lambda x,y,t: self.k*(self.theta-y)
        volH_2_y = lambda x,y,t: self.xi*np.sqrt(y)
        _, H, volatility_H = euler_scheme(self.T,self.dt,self.Npaths,True,x0 = self.S0,drift_x = drift_H_x,
                                    vol1_x = volH_1_x,W1 = self.W,
                                    drift_y = drift_H_y,vol2_y = volH_2_y,y0 = self.v0, W2 = self.__corrBM())
        self.volatility_H = volatility_H
        self.H = H

    def incr(self):
        if self.H is not None:
            return np.diff(self.H)
        else:
            raise ValueError("The paths have not been generated yet. Please run the 'generate' method first.")

    def time(self):
        return np.linspace(0, self.T, self.N+ 1)
    def plot(self):
        for j in range(self.Npaths):
            plt.plot(self.time(), self.H[j, :])
            plt.title("Heston Model Trajectories")
            plt.xlabel("Time")
            plt.ylabel("Price")
            plt.show()

In [ ]:
class CEVModel(object):
    def __init__(self,Npaths,T,dt,mu=0.04,sigma=0.2,gamma=0.7,S0=1):
        self.T = T
        self.mu = mu
        self.sigma = sigma
        self.gamma = gamma
        self.N = int(T/dt)
        self.dt = dt
        self.Npaths = Npaths
        self.W = None #Paths of BM with discretization int(T/dt) and Npaths different paths
        self.S0_scalar = S0
        self.S0 = S0*np.ones((Npaths,)) #initial price vector
        self.CEV = None
    def __generateBM(self, training_seed = None, eval_seed = None):
        if training_seed is not None:
            np.random.seed(training_seed)
        if eval_seed is not None:
            np.random.seed(eval_seed)
        dW =  np.sqrt(self.dt)*np.random.randn(self.N*self.Npaths).reshape((self.Npaths,self.N))
        W = np.concatenate((np.zeros((self.Npaths,1)),np.cumsum(dW,axis = 1)),axis =1)
        return W
    def generate(self,training_seed = None, eval_seed = None):
        self.W = self.__generateBM(training_seed, eval_seed)
        drift_CEV_x = lambda x,t: self.mu*x
        volCEV_1_x = lambda x,t: self.sigma*x**self.gamma
        _,CEV,_ = euler_scheme(self.T,self.dt,self.Npaths,False,self.S0,drift_CEV_x,volCEV_1_x,self.W)
        self.CEV = CEV
    def incr(self):
        if self.CEV is not None:
            return np.diff(self.CEV)
        else:
            raise ValueError("The paths have not been generated yet. Please run the 'generate' method first.")

    def time(self):
        return np.linspace(0, self.T, self.N+ 1)

    def plot(self):
        for j in range(self.Npaths):
            plt.plot(self.time(), self.CEV[j, :])
            plt.title("CEV Model Trajectories")
            plt.xlabel("Time")
            plt.ylabel("Price")
            plt.show()

In the following, we specify our utility function. We use exponential utility
$$u: [0,\infty) \to [0,\infty), x \mapsto \frac{1}{\alpha}(1-\exp{-\alpha x})$$
with $\alpha = 1.5$. This corresponds to a risk-averse person. Clearly, this parameter can be adjusted.

In [ ]:
alpha = 1.5
if alpha <= 0 :
    ValueError('Parameter alpha must be greater than 0.')
#Definition of power utility
def utility(x):
    return 1/alpha*(1-torch.exp(-alpha*x))#(x**(1-eta))/(1-eta)

In the following, we setup the class ```MarkovianModel``` that initializes list of ```N``` (depends on time discretization) neural networks with some number of layers (default = 3) and hidden_size (default = 32). The forward method then computes at each time step $j$ the current wealth using the $\alpha_j$ obtained from the neural network at time $j$, which takes as input wealth and price at time $j-1$. It is important to mention that we add ```torch.clamp(log_wealth,min = -80, max = 80)``` to prevent overflow.

In [ ]:
class MarkovianModel(nn.Module):
    def __init__(self, num_layers=3, hidden_size=32, r=None, T=None, dt=None, device=None):
        super(MarkovianModel, self).__init__()
        self.num_layers = num_layers
        self.hidden_size = hidden_size
        self.N = int(T/dt)
        self.r = r
        self.T = T
        self.device = device if device is not None else torch.device('cpu')

        self.networks = nn.ModuleList()
        for j in range(self.N):
            layers = []
            layers.append(nn.Linear(2, hidden_size))
            layers.append(nn.Tanh())
            for _ in range(num_layers - 2):
                layers.append(nn.Linear(hidden_size, hidden_size))
                layers.append(nn.Tanh())
            layers.append(nn.Linear(hidden_size, 1))
            layers.append(nn.Tanh())
            network = nn.Sequential(*layers)
            for layer in network:
                    if isinstance(layer, nn.Linear):
                        # Xavier initialization for tanh
                        nn.init.xavier_uniform_(layer.weight, gain=nn.init.calculate_gain('tanh'))
                        # Bias initialization
                        if layer.bias is not None:
                            nn.init.zeros_(layer.bias)
            self.networks.append(network)
        self.to(self.device)

    def forward(self, price, wealth, incr_list, return_histories=False, sigma=None, volatility_H=None, gamma=None):
        price = price.to(self.device)
        wealth = wealth.to(self.device)
        incr_list = incr_list.to(self.device)
        if volatility_H is not None:
            volatility_H = volatility_H.to(self.device)
        batch_size = len(incr_list)
        if return_histories:
            price_history = torch.zeros((batch_size, self.N + 1, 1), device=self.device)
            wealth_history = torch.zeros((batch_size, self.N + 1, 1), device=self.device)
            alpha_history = torch.zeros((batch_size, self.N + 1, 1), device=self.device)
            price_history[:, 0, 0] = price.squeeze()
            wealth_history[:, 0, 0] = wealth.squeeze()
            alpha_history[:, 0, 0] = 0.0
        for j in range(self.N):
            network_input = torch.cat([
                price.view(batch_size, 1),
                wealth.view(batch_size, 1)
            ], dim=1)
            strategy = self.networks[j](network_input)
            #print(f'This is the strategy {strategy}')
            incr = incr_list[:, j].view(batch_size, 1)
            #print(f'This is the wealth {wealth}.')
            #wealth = wealth*(1+(1-strategy)*self.r*self.T/self.N+ strategy/price*incr)
            log_wealth = torch.log(wealth)
            #print(f'This is the {log_wealth} after clamp.')
            log_wealth = log_wealth + (1- strategy)*(self.r * self.T / self.N)+ strategy * incr / price
            #print(f'This is the log wealth before taking subtracting the model dependent part {log_wealth}')
            if sigma is not None and gamma is None:
                log_wealth = log_wealth - (strategy ** 2) * (sigma ** 2 / 2) * (self.T / self.N)
            elif volatility_H is not None:
                #print(f'This is the shape {volatility_H[:, j].shape} and {volatility_H[:, j].view(batch_size,1).shape}.')
                log_wealth = log_wealth - (strategy ** 2) * (volatility_H[:, j].view(batch_size, 1) / 2) * (self.T / self.N)
            elif gamma is not None:
                log_wealth = log_wealth - (strategy ** 2) * (sigma ** 2 * price ** (gamma - 2) / 2) * (self.T / self.N)
            # #print(f'This is the log wealth before taking exponential {log_wealth}')
            wealth = torch.exp(log_wealth)
            price = price + incr
            if return_histories:
                price_history[:, j + 1, 0] = price.squeeze()
                wealth_history[:, j + 1, 0] = wealth.squeeze()
                alpha_history[:, j + 1, 0] = strategy.squeeze()
        #print(f'This is the wealth{wealth},strategy {strategy} and price {price}.')
        if torch.isnan(wealth).any() or torch.isnan(strategy).any() or torch.isnan(price).any():
            print("Warning: NaN detected in wealth, strategy, or price tensor.")
        if return_histories:
            return wealth, price_history, wealth_history, alpha_history
        else:
            return wealth


Similarly to the previous approach, we now define a class for training an agent that incorporates historical information. In this setup, the forward function receives not only the current price and wealth but also their past values as input. Importantly, we restrict the input history to a configurable lookback window, which helps manage the input dimensionality, especially when using fine time discretizations.

In [ ]:
class PathDependentTradingStrategy(nn.Module):
    def __init__(self, lookback_window=1, hidden_size=32, num_layers=3, r=None, T=None, dt=None, device=None):
        """
        Path-dependent trading strategy that considers the entire history of prices and wealth.
        """
        super(PathDependentTradingStrategy, self).__init__()
        self.lookback_window = lookback_window
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.r = r
        self.T = T
        self.dt = dt
        self.N = int(T/dt)
        self.device = device if device is not None else torch.device('cpu')
        self.max_lookback = min(lookback_window, self.N)
        
        # Networks for each time step
        self.networks = nn.ModuleList()
        for j in range(self.N):
            layers = []
            layers.append(nn.Linear(2*min(j+1,self.max_lookback-1) + 2, hidden_size)) # First layer takes history (capped by lookback_window) + current price + current wealth
            layers.append(nn.Tanh())

            for _ in range(num_layers - 2):
                layers.append(nn.Linear(hidden_size, hidden_size))
                layers.append(nn.Tanh())

            layers.append(nn.Linear(hidden_size, 1))
            layers.append(nn.Tanh())

            # Create a sequential network for this time step
            network = nn.Sequential(*layers)

            # Initialize weights
            for layer in network:
                    if isinstance(layer, nn.Linear):
                        # Xavier initialization for tanh
                        nn.init.xavier_uniform_(layer.weight, gain=nn.init.calculate_gain('tanh'))
                        # Bias initialization
                        if layer.bias is not None:
                            nn.init.zeros_(layer.bias)

            self.networks.append(network.to(self.device))

    def forward(self, initial_price, initial_wealth, incr_list, sigma=None, volatility_H=None, gamma=None, return_histories=False):
        # Move all tensors to the correct device
        device = self.device
        initial_price = initial_price.to(device)
        initial_wealth = initial_wealth.to(device)
        incr_list = incr_list.to(device)
        if volatility_H is not None:
            volatility_H = volatility_H.to(device)

        batch_size = incr_list.shape[0]
        price = initial_price
        wealth = initial_wealth

        if return_histories:
            price_history = torch.zeros((batch_size, self.N + 1, 1), device=device)
            wealth_history = torch.zeros((batch_size, self.N + 1, 1), device=device)
            alpha_history = torch.zeros((batch_size, self.N + 1, 1), device=device)
            price_history[:, 0, 0] = initial_price.squeeze()
            wealth_history[:, 0, 0] = initial_wealth.squeeze()
            alpha_history[:, 0, 0] = 0.0

        price_buffer = torch.zeros((batch_size, self.max_lookback), device=device)
        wealth_buffer = torch.zeros((batch_size, self.max_lookback), device=device)
        price_buffer[:, 0] = price.view(-1)
        wealth_buffer[:, 0] = wealth.view(-1)

        for j in range(self.N):
            buffer_idx = j % self.max_lookback
            lookback = min(j + 1, self.max_lookback-1)
            ind = [(buffer_idx - i) % self.max_lookback for i in range(lookback)]
            ind.reverse()
            p_history = price_buffer[:, ind]
            w_history = wealth_buffer[:, ind]
            #print(f'This is the price {price} and the price history fed in {p_history}.')
            network_input = torch.cat([
                p_history,
                w_history,
                price.view(batch_size, 1),
                wealth.view(batch_size, 1)
            ], dim=1)
            # print('Dimension Price', p_history.shape)
            # print('Input',network_input.shape)
            # print('NN',self.networks[j])
            strategy = self.networks[j](network_input)
            incr = incr_list[:, j].view(batch_size, 1)
            #print(f'This is the wealth {wealth}.')
            log_wealth = torch.log(wealth)
            log_wealth = log_wealth + (1-strategy)*(self.r * self.T / self.N) + strategy * incr / price
            if sigma is not None and gamma is None:
                log_wealth = log_wealth - (strategy ** 2) * (sigma**2 / 2) * (self.T / self.N)
            elif volatility_H is not None:
                log_wealth = log_wealth - (strategy ** 2) * (volatility_H[:, j].view(batch_size, 1) / 2) * (self.T / self.N)
            elif gamma is not None:
                log_wealth = log_wealth - (strategy ** 2) * (sigma**2 * price**(gamma-2)/ 2) * (self.T / self.N)
            wealth = torch.exp(log_wealth)
            price = price + incr
            next_idx = (buffer_idx + 1) % self.max_lookback
            price_buffer[:, next_idx] = price[:, 0]
            wealth_buffer[:, next_idx] = wealth[:, 0]
            #alpha_buffer[:, next_idx, 0] = strategy[:, 0]
            if return_histories:
                price_history[:, j+1, 0] = price[:, 0]
                wealth_history[:, j+1, 0] = wealth[:, 0]
                alpha_history[:, j+1, 0] = strategy[:, 0]
               
        if return_histories:
            return wealth, price_history, wealth_history, alpha_history
        else:
            return wealth


In the following class we specify our loss function, provide methods that generate the data, train the neural network and evaluate it. Moreover, a function to plot the results is defined.

In [ ]:
class TradingModelTrainer:
    def __init__(self, model, price_model_str, T, dt, lr=0.001,initial_wealth = 1, utility_func=utility, device=None):
        self.device = device if device is not None else ('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = model.to(self.device)
        self.optimizer = optim.Adam(self.model.parameters(), lr=lr)
        self.price_model_str = price_model_str  # either 'BS', 'H' or 'CEV'
        self.utility_func = utility_func
        self.T = T
        self.dt = dt
        self.price_model = None
        self.initial_wealth = initial_wealth
    def loss_function(self, terminal_wealth):
        utility_vals = self.utility_func(terminal_wealth)
        #print(f'This are the utility_vals {utility_vals}')
        return -torch.mean(utility_vals) # negative expected utility to be minimized since utility is positive
    def generate_training_data(self, batch_size, training_seed = None, eval_seed = None):
        # First we need to create price_model
        if self.price_model_str == 'BS':
            self.price_model = BSModel(batch_size, self.T, self.dt)
            self.price_model.generate(training_seed,eval_seed)
        elif self.price_model_str == 'H':
            self.price_model = HestonModel(batch_size, self.T, self.dt)
            self.price_model.generate(training_seed,eval_seed)
        elif self.price_model_str == 'CEV':
            self.price_model = CEVModel(batch_size, self.T, self.dt)
            self.price_model.generate(training_seed,eval_seed)
        else:
            raise ValueError(f"Unknown price model: {self.price_model_str}")

        # Initial conditions
        price = self.price_model.S0_scalar*torch.ones(batch_size, 1, dtype=torch.float32, device=self.device).view(-1, 1)
        wealth = self.initial_wealth*torch.ones(batch_size, 1, dtype=torch.float32, device=self.device).view(-1, 1)
        incr_list = torch.tensor(self.price_model.incr(), dtype=torch.float32, device=self.device)
        return price, wealth, incr_list

    def train(self, num_epochs=10, batch_size=100, verbose=True):
        losses = []
        # Generate Training Data
        price, wealth, incr_list = self.generate_training_data(batch_size = batch_size,training_seed = 517)
        for epoch in range(num_epochs):
            epoch_loss = 0
            self.optimizer.zero_grad()
            # Forward pass
            if self.price_model_str == 'BS':
                terminal_wealth= self.model(price, wealth, incr_list, sigma=self.price_model.sigma, return_histories=False)
            elif self.price_model_str == 'H':
                terminal_wealth = self.model(price, wealth, incr_list, volatility_H=torch.tensor(self.price_model.volatility_H, dtype=torch.float32, device=self.device), return_histories=False)
            elif self.price_model_str == 'CEV':
                terminal_wealth = self.model(price, wealth, incr_list, gamma=self.price_model.gamma, sigma=self.price_model.sigma, return_histories=False)
            # Calculate loss
            loss = self.loss_function(terminal_wealth)
            #print(f'This is the loss {loss}')
            # Backward pass and optimization
            loss.backward()
            self.optimizer.step()
            epoch_loss += loss.item()
            losses.append(epoch_loss)
            if verbose:
                print(f'Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.6f}')
        return losses

    def evaluate_model(self, num_simulations=100):
        self.model.eval()  # Set model to evaluation mode
        with torch.no_grad():
            # Generate evaluation data
            price, wealth, incr_list = self.generate_training_data(num_simulations,eval_seed =121)
            # Forward pass
            if self.price_model_str == 'BS':
                terminal_wealth, price_history, wealth_history, alpha_history = self.model(
                    price, wealth, incr_list, sigma=self.price_model.sigma, return_histories=True)
            elif self.price_model_str == 'H':
                terminal_wealth, price_history, wealth_history, alpha_history = self.model(
                    price, wealth, incr_list, volatility_H=torch.tensor(self.price_model.volatility_H, dtype=torch.float32, device=self.device), return_histories=True)
            elif self.price_model_str == 'CEV':
                terminal_wealth, price_history, wealth_history, alpha_history = self.model(
                    price, wealth, incr_list, gamma=self.price_model.gamma, sigma=self.price_model.sigma, return_histories=True)
            utility_vals = self.utility_func(terminal_wealth)
            # Move to CPU for numpy conversion
            mean_wealth = terminal_wealth.mean().item()
            mean_utility = utility_vals.mean().item()
            wealth_distribution = terminal_wealth.cpu().numpy().flatten()
            price_history = price_history.cpu().numpy() if hasattr(price_history, 'cpu') else price_history
            wealth_history = wealth_history.cpu().numpy() if hasattr(wealth_history, 'cpu') else wealth_history
            alpha_history = alpha_history.cpu().numpy() if hasattr(alpha_history, 'cpu') else alpha_history
            return mean_wealth, mean_utility, wealth_distribution, price_history, wealth_history, alpha_history


def plot_wealth_history(T, wealth, Title=None):
    plt.figure(figsize=(8, 4))
    for j in range(len(wealth)):
        t = np.linspace(0, T, len(wealth[0, :, 0]))
        plt.plot(t, wealth[j, :, 0])
    plt.title('Wealth History')
    plt.xlabel('Time')
    plt.ylabel('Wealth')
    if Title is not None:
        plt.suptitle(Title, fontsize=14)
    plt.tight_layout()
    plt.show()

def plot_price_history(T, price, Title=None):
    plt.figure(figsize=(8, 4))
    for j in range(len(price)):
        t = np.linspace(0, T, len(price[0, :, 0]))
        plt.plot(t, price[j, :, 0])
    plt.title('Price History')
    plt.xlabel('Time')
    plt.ylabel('Price')
    if Title is not None:
        plt.suptitle(Title, fontsize=14)
    plt.tight_layout()
    plt.show()

def plot_alpha_history(T, alpha, Title=None):
    plt.figure(figsize=(8, 4))
    t = np.linspace(0, T, len(alpha[0]))
    plt.plot(t, alpha[0, :, 0])
    plt.title('Proportion')
    plt.xlabel('Time')
    plt.ylabel('Alpha')
    if Title is not None:
        plt.suptitle(Title, fontsize=14)
    plt.tight_layout()
    plt.show()

def plot_wealth_distribution(wealth_distribution, Title=None):
    plt.figure(figsize=(8, 4))
    plt.hist(wealth_distribution, bins=50)
    plt.title('Terminal Wealth Distribution')
    plt.xlabel('Terminal Wealth')
    plt.ylabel('Frequency')
    if Title is not None:
        plt.suptitle(Title, fontsize=14)
    plt.tight_layout()
    plt.show()

def compare_wealth_distributions(wealth_distribution1, wealth_distribution2, label1='Markov', label2='PathDependent', Title=None):
    plt.figure(figsize=(8, 4))
    plt.hist([wealth_distribution1, wealth_distribution2], bins=40, label=[label1, label2], color=['blue', 'orange'], alpha=0.7, edgecolor='black')
    # Calculate stats
    mean1 = np.mean(wealth_distribution1)
    std1 = np.std(wealth_distribution1)
    mean2 = np.mean(wealth_distribution2)
    std2 = np.std(wealth_distribution2)
    # Plot mean lines
    plt.axvline(mean1, color='blue', linestyle='dashed', linewidth=2, label=f'{label1} Mean: {mean1:.3f}')
    plt.axvline(mean2, color='orange', linestyle='dashed', linewidth=2, label=f'{label2} Mean: {mean2:.3f}')
    # Plot std lines and add to legend
    plt.axvline(mean1 - std1, color='blue', linestyle='solid', linewidth=1, alpha=0.7, label=f'{label1} Std: {std1:.3f}')
    plt.axvline(mean1 + std1, color='blue', linestyle='solid', linewidth=1, alpha=0.7)
    plt.axvline(mean2 - std2, color='orange', linestyle='solid', linewidth=1, alpha=0.7, label=f'{label2} Std: {std2:.3f}')
    plt.axvline(mean2 + std2, color='orange', linestyle='solid', linewidth=1, alpha=0.7)
    plt.title('Comparison of Terminal Wealth Distributions')
    plt.xlabel('Terminal Wealth')
    plt.ylabel('Frequency')
    plt.legend()
    if Title is not None:
        plt.suptitle(Title, fontsize=14)
    plt.tight_layout()
    plt.show()


In [ ]:
def train_markovian_model(num_layers=3, n=32, r=0.02, T=1.0, dt=0.01,initial_wealth = 1, path_model=None):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = MarkovianModel(num_layers=num_layers, hidden_size=n, r=r, T=T, dt=dt, device=device)
    trainer = TradingModelTrainer(model, price_model_str=path_model, lr=0.001,initial_wealth = initial_wealth, T=T, dt=dt, device=device)
    _ = trainer.train(num_epochs=100, batch_size=10000) 
    return trainer


In [ ]:
def train_path_dependent_model(num_layers=3, n=32, lw=50, r=0.02, T=1.0, dt=0.01,initial_wealth = 1, path_model=None): #maximal history are lw-1 timesteps
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = PathDependentTradingStrategy(num_layers=num_layers, hidden_size=n, lookback_window=lw, r=r, T=T, dt=dt, device=device)
    trainer = TradingModelTrainer(model, price_model_str=path_model, lr=0.001,initial_wealth = initial_wealth, T=T, dt=dt, device=device)
    _ = trainer.train(num_epochs=100, batch_size=1) 
    return trainer


## Training

In [ ]:
#Runtime on my mac less then 2-3 minutes per task

In [ ]:
trainer_BS = train_markovian_model(path_model='BS') 

In [ ]:
trainer_BS_PD = train_path_dependent_model(path_model='BS') 

In [ ]:
trainer_H = train_markovian_model(path_model='H') 

In [ ]:
trainer_H_PD = train_path_dependent_model(path_model='H') 

In [ ]:
trainer_CEV = train_markovian_model(path_model='CEV') 

In [ ]:
trainer_CEV_PD = train_path_dependent_model(path_model='CEV') 

## Testing

In [ ]:
mean_wealth_BS_M, mean_utility_BS_M, wealth_distribution_BS_M,price_history_BS_M,wealth_history_BS_M,alpha_history_BS_M = trainer_BS.evaluate_model(num_simulations=1000)
print('Mean Wealth BS of Markovian Agent:', mean_wealth_BS_M)
print('Mean utility BS of Markovian Agent:', mean_utility_BS_M)

In [ ]:
mean_wealth_BS_PD, mean_utility_BS_PD, wealth_distribution_BS_PD,price_history_BS_PD,wealth_history_BS_PD,alpha_history_BS_PD = trainer_BS_PD.evaluate_model(num_simulations=1000)
print('Mean Wealth BS of Path Dependent Agent:', mean_wealth_BS_PD)
print('Mean utility BS of Path Dependent Agent:', mean_utility_BS_PD)


In [ ]:
compare_wealth_distributions(wealth_distribution1=wealth_distribution_BS_M,wealth_distribution2=wealth_distribution_BS_PD,Title = 'Black-Scholes')

In [ ]:
mean_wealth_H_M, mean_utility_H_M, wealth_distribution_H_M,price_history_H_M,wealth_history_H_M,alpha_history_H_M = trainer_H.evaluate_model(num_simulations=1000)
print('Mean Wealth Heston Model of Markovian Agent:', mean_wealth_H_M)
print('Mean utility Heston Model of Markovian Agent:', mean_utility_H_M)


In [ ]:
mean_wealth_H_PD, mean_utility_H_PD, wealth_distribution_H_PD,price_history_H_PD,wealth_history_H_PD,alpha_history_H_PD = trainer_H_PD.evaluate_model(num_simulations=1000)
print('Mean Wealth Heston of Path Dependent Agent:', mean_wealth_H_PD)
print('Mean utility Heston of Path Dependent Agent:', mean_utility_H_PD)


In [ ]:
compare_wealth_distributions(wealth_distribution1=  wealth_distribution_H_M,wealth_distribution2=wealth_distribution_H_PD,Title = 'Heston Model')

In [ ]:
mean_wealth_CEV_M, mean_utility_CEV_M, wealth_distribution_CEV_M,price_history_CEV_M,wealth_history_CEV_M,alpha_history_CEV_M = trainer_CEV.evaluate_model(num_simulations=1000)
print('Mean Wealth CEV Model of Markovian Agent:', mean_wealth_CEV_M)
print('Mean utility CEV Model of Markovian Agent:', mean_utility_CEV_M)

In [ ]:
mean_wealth_CEV_PD, mean_utility_CEV_PD, wealth_distribution_CEV_PD,price_history_CEV_PD,wealth_history_CEV_PD,alpha_history_CEV_PD = trainer_CEV_PD.evaluate_model(num_simulations=1000)
print('Mean Wealth CEV Model of Path Dependent Agent:', mean_wealth_CEV_PD)
print('Mean utility CEV Model of Path Dependent Agent:', mean_utility_CEV_PD)


In [ ]:
compare_wealth_distributions(wealth_distribution1=wealth_distribution_CEV_M,wealth_distribution2=wealth_distribution_CEV_PD,Title = 'CEV Model')

## Conclusion
We observe that the two agents perform equally for the **Black-Scholes** Model in terms of mean terminal wealth. This is because the wealth process $X$ in the Black-Scholes model is a *Markov process* controlled by $X$. Indeed, plugging the dynamics of $S$ into the SDE (of the wealth) introduced earlier yields
$$ \mathrm{d}X_t = \alpha_t X_t\left(\mu \mathrm{d}t +\sigma \mathrm{d}W_t\right) + (1-\alpha_t)X_t r\mathrm{d}t.$$
For the **CEV** model, we have the following dynamics for the wealth process
$$ \mathrm{d}X_t = \alpha_t X_t\left(\mu \mathrm{d}t +\sigma S_t^{\gamma-2}\mathrm{d}W_t\right) + (1-\alpha_t)X_t r\mathrm{d}t.$$
Hence, the change in wealth is controlled not only by the previous wealth but also by the previous stock price $S_t$. Nevertheless, since our markovian trading agent tries to find a strategy depending on time, current wealth and current stock price there should not be a significant advantage in observing the entire past (or a lookback window). In reality, we can even observe a slightly better perfomance of the markovian trading agent in terms of mean terminal wealth. This might be due to the fact that the architecture of the neural net for the path dependent agent can be improved using LSTM/RNN and not just increasing input size.

Further, we observe a small difference between the two agents when using the **Heston** model. The dynamics of the wealth process is
$$ \mathrm{d}X_t = \alpha_t X_t\left(\mu \mathrm{d}t +\sqrt{\nu_t}\mathrm{d}W_t\right) + (1-\alpha_t)X_t r\mathrm{d}t,$$
where $\nu_t$ is the volatility process. Thus, the wealth also depends on the volatility process $\nu$. In particular, $\nu$ cannot directly be observed in the market and has to be estimated from the past. Consequently, in the Heston case the agent that takes into account the whole path (or a lookback window) outperforms the markovian trading agent. I think the difference would be bigger if we would use a more advanced architecture for the path dependent trading agent.

Furthermore, there is a difference between the standard deviations in the wealth distributions of the two agents. We observe that the standard deviation in the terminal wealth of the path dependent agent is typically bigger or equal than the one arising from the terminal wealth of the markovian agent. A possible reason is that the former gets significanlty more input data and hence the results are wider spread.